vSphere Infrastructure Deployment and Management Technical Reference

This reference document outlines the architectural standards, configuration protocols, and operational workflows required to maintain a high-availability VMware vSphere environment. It is intended for senior systems architects to ensure infrastructure consistency and performance.


--------------------------------------------------------------------------------


1. Architectural Foundations: ESXi and vCenter Server

The vSphere ecosystem relies on a decoupled architecture where the compute layer (ESXi) is abstracted from the management plane (vCenter Server). This separation allows for massive scalability and the centralization of complex logic, such as resource scheduling and high availability.

The ESXi Host: Bare-Metal Hypervisor

The ESXi host is a Type-1, bare-metal hypervisor installed directly onto hardware. Because it operates without a general-purpose underlying OS, overhead is minimized.

* Datastore Visibility Threshold: A critical hardware constraint exists for local storage: an ESXi host will not display a local datastore unless the disk capacity exceeds 128GB.
* Processor Definition: In the vSphere context, a "Socket" is defined as a physical processor (CPU) package. Logic and licensing often scale based on this physical count rather than core count.
* Management Entry Points: Local host management is conducted via the Direct Console User Interface (DCUI)—the "yellow and black" physical terminal—or the ESXi Host Client (Web UI).

Management Interfaces

Feature	ESXi Host Client	vSphere Client
Access Method	Direct (Host IP/FQDN)	Centralized (vCenter Server)
Scope	Single Host Management	Cluster/Data Center Management
Primary Use Case	Initial setup, troubleshooting, and emergency access when vCenter is offline.	Day-to-day operations, VM migrations, and advanced feature configuration.
Authority	Direct control over local resources.	Centralized control; VDS settings are Read-Only at the host level.

A stabilized host layer provides the immutable foundation upon which the centralized management plane is built.


--------------------------------------------------------------------------------


2. vCenter Server Management and Appliance Lifecycle

vCenter Server is the intelligence of the data center. It organizes infrastructure into a hierarchical structure, utilizing Datacenter objects as the primary containers for hosts, clusters, and storage.

vCenter Server Appliance (vCSA) Architecture

The vCSA is a pre-configured virtual machine running on VMware Photon OS (a security-hardened Linux distribution).

* Deployment Method: Modern vCenter deployment has shifted from OVF/OVA templates to an ISO-based installer.
* Installation Stages: Installation occurs in two stages: the initial deployment of the appliance followed by the configuration of the Single Sign-On (SSO) domain.

Lifecycle and Disaster Recovery (VAMI)

The vCenter Server Management Interface (VAMI), accessible via Port 5480, provides appliance-level health data and update management.

* Backup Methodologies:
  * Image-based: Captures the entire VM state (OS and configuration).
  * File-based (VAMI): Backs up configuration and database files only. This requires a protocol such as FTP.
* Restore Workflow: To restore from a file-based backup, run the vCenter installer in Restore Mode.
  * Architect’s Note: The original (failed) vCenter must be powered off before starting the restore to prevent IP and hostname conflicts.

Security: Lockdown Mode

Lockdown Mode prevents unauthorized direct access to hosts. The following matrix defines access restrictions:

Mode	DCUI Access	ESXi Host Client Access	vSphere Client Access
Normal	Allowed	Disabled	Allowed
Strict	Disabled	Disabled	Allowed

Once the management plane is secured and the organizational hierarchy is defined, the deployment of virtual workloads can proceed.


--------------------------------------------------------------------------------


3. Virtual Machine (VM) Configuration and File Anatomy

VMs are encapsulated as a collection of files residing on a datastore. This encapsulation ensures workload portability across the vSphere cluster.

Technical Glossary of VM Files

* .vmx: The primary configuration file (RAM, CPU, NIC settings). The VM cannot power on without this file.
* .vmdk: The virtual disk descriptor and data file.
* .vmem: A backup of the VM’s main memory; used specifically during snapshots and backups.
* .vswp: The memory swap file. Created when the VM powers on (matching the size of unreserved RAM) and deleted upon power-off.
* .nvram: Stores the VM’s BIOS/EFI state.
* .log: Tracks VM events and execution history.
* .vmsd / .vmsn: Snapshot metadata and state files.
* .vmss: Used only when a VM is in a "Suspended" state.

Storage Provisioning and Capacity Planning

* Thin Provisioning: Consumes physical space only as data is written.
* Thick Provisioning: Immediately allocates the full disk capacity on the datastore.
* Warning: When removing storage from a VM, the system provides two options: "Remove from virtual machine" or "Delete from disk." Choosing only to remove it from the VM leaves the .vmdk as an orphaned file, consuming datastore space without visibility.

OVF and OVA Formats

* OVA: A single, consolidated file (similar to an ISO).
* OVF: A folder containing multiple component files (descriptor, disks, manifest).


--------------------------------------------------------------------------------


4. Virtual Networking Architecture and Management

vSphere abstracts physical networking into logical switches to facilitate traffic isolation and VM communication.

Core Networking Components

1. VM Port Groups: Connection points for VM traffic. These define network membership and VLAN tagging.
2. VMkernel Ports: Dedicated interfaces for ESXi system services (Management, vMotion, Storage). Every VMkernel port requires a dedicated IP address.
3. Uplinks: The physical network adapters (NICs) on the host.

Virtual Switch Comparison

* Standard Switch (VSS): Configured and managed at the individual host level.
* Distributed Switch (VDS): Centrally managed via vCenter. It provides consistent configuration across all hosts in a cluster.

Architect’s Note on VDS Deployment: When assigning physical adapters (Uplinks) to a VDS, those adapters must be new/unused. Do not attempt to assign an uplink already actively carrying production traffic for a VSS without a migration plan.

Migration Workflow: VSS to VDS

To move services from a VSS to a VDS, use the "Manage Host" networking function. This allows for the orderly migration of VMkernel ports and VM port groups to the distributed switch. To protect against VDS failure, architects must manually use the "Export Configuration" feature to create a compressed backup of the network logic.


--------------------------------------------------------------------------------


5. Storage Connectivity and Datastore Standardization

vSphere supports Block (SAN/DAS) and File (NAS) storage. Standardizing storage is essential for enabling advanced vSphere features like vMotion and DRS.

Datastore Standards

* VMFS: A high-performance cluster file system for block storage (iSCSI, Fibre Channel).
* NFS: File-level storage (NAS).
  * Architect’s Note: NFS datastores are managed as shared folders at the storage level and cannot be formatted from the vSphere client.
* vSAN / vVols: Advanced software-defined storage architectures.

iSCSI Configuration Workflow

1. Manual Step: The Software iSCSI Adapter must be manually added to the ESXi host before discovery can begin.
2. Connectivity: Configure the iSCSI Initiator (on the host) to point to the iSCSI Target (on the storage system) via Static or Dynamic discovery.
3. Rescanning: Once configured, a storage rescan is required for the LUNs to become visible.

Critical Warning: Storage Traffic Routing

By default, if a dedicated VMkernel port is not configured for storage traffic, vSphere will attempt to route storage data through the Management VMkernel port. This creates significant performance bottlenecks and security risks.

* Best Practice: Always isolate storage traffic on a dedicated subnet and VLAN. For high-availability, use two VMkernel ports on separate physical adapters, ensuring storage traffic is never multiplexed across the same interface simultaneously.


--------------------------------------------------------------------------------


6. Operational Mobility: VM Migration and Templates

Mobility is the cornerstone of vSphere's resiliency, allowing VMs to traverse physical hardware without downtime.

Technical Migration Matrix

Migration Type	VM State	CPU Requirement	Storage Requirement
Cold Migration	Powered Off	Supports different vendors	Can move across datastores
Suspended	Paused	Must be identical architecture	Shared storage required
vMotion	Powered On	Must be same CPU Vendor	Shared storage required
Storage vMotion	Powered On	Same Vendor/Host	Moves between datastores
Shared-Nothing	Powered On	Must be same CPU Vendor	Moves host and storage
Cross vCenter	Powered On	Compatible architectures	vCenter-to-vCenter link

Deployment Efficiency: Templates and Cloning

* Templates: A master image used for mass deployment.
  * Cloning to Template: Use this workflow if you need to create a template from a VM that is currently powered on.
* Cloning: Creates an exact replica of a VM (including MAC and UUID).
* Guest OS Customization: To prevent IP and Hostname conflicts, always apply a Customization Specification (Sysprep) during deployment from a template.

The adherence to these deployment and management standards ensures a resilient, enterprise-grade vSphere environment capable of handling high-demand production workloads.
